In [12]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.window import Window
from pyspark.sql.functions import * 

In [4]:
sliding_sess=SparkSession.builder.appName("sliding_window").getOrCreate()
spark_df=sliding_sess.read.csv("train.csv",header=True,inferSchema=True)

**Sliding Window**

In [6]:
spark_df.show(5)

+-----+---------+----------+-----+---------+----+-----+------------+-------------+
|Store|DayOfWeek|      Date|Sales|Customers|Open|Promo|StateHoliday|SchoolHoliday|
+-----+---------+----------+-----+---------+----+-----+------------+-------------+
|    1|        5|2015-07-31| 5263|      555|   1|    1|           0|            1|
|    2|        5|2015-07-31| 6064|      625|   1|    1|           0|            1|
|    3|        5|2015-07-31| 8314|      821|   1|    1|           0|            1|
|    4|        5|2015-07-31|13995|     1498|   1|    1|           0|            1|
|    5|        5|2015-07-31| 4822|      559|   1|    1|           0|            1|
+-----+---------+----------+-----+---------+----+-----+------------+-------------+
only showing top 5 rows



In [16]:
sliding_window=Window.partitionBy("Store").orderBy("Date").rowsBetween(-2,2)
spark_df=spark_df.withColumn("moving_avg_sales", avg("Sales").over(sliding_window))

In [ ]:
# spark_df=spark_df.drop("sliding_avg_sales")
spark_df.show()
print(spark_df.printSchema())

+-----+---------+----------+-----+---------+----+-----+------------+-------------+----------------+
|Store|DayOfWeek|      Date|Sales|Customers|Open|Promo|StateHoliday|SchoolHoliday|moving_avg_sales|
+-----+---------+----------+-----+---------+----+-----+------------+-------------+----------------+
|   28|        2|2013-01-01|    0|        0|   0|    0|           a|            1|          3415.0|
|   28|        3|2013-01-02| 4958|      609|   1|    0|           0|            1|          3931.5|
|   28|        4|2013-01-03| 5287|      641|   1|    0|           0|            1|          3559.2|
|   28|        5|2013-01-04| 5481|      623|   1|    0|           0|            1|          3559.2|
|   28|        6|2013-01-05| 2070|      287|   1|    0|           0|            0|          4592.8|
|   28|        7|2013-01-06|    0|        0|   0|    0|           a|            0|          5113.8|
|   28|        1|2013-01-07|10126|      989|   1|    1|           0|            0|          5344.0|


In [19]:
spark_df.describe()

DataFrame[summary: string, Store: string, DayOfWeek: string, Sales: string, Customers: string, Open: string, Promo: string, StateHoliday: string, SchoolHoliday: string, sliding_avg_sales: string, moving_avg_sales: string]

**Fixed Size Window**

In [22]:
fixed_size_window=Window.partitionBy("Store").orderBy("Date").rowsBetween(-3,0)
spark_df=spark_df.withColumn("fixed_size_avg_sales", avg("Sales").over(fixed_size_window))
spark_df.show()

+-----+---------+----------+-----+---------+----+-----+------------+-------------+----------------+--------------------+
|Store|DayOfWeek|      Date|Sales|Customers|Open|Promo|StateHoliday|SchoolHoliday|moving_avg_sales|fixed_size_avg_sales|
+-----+---------+----------+-----+---------+----+-----+------------+-------------+----------------+--------------------+
|   28|        2|2013-01-01|    0|        0|   0|    0|           a|            1|          3415.0|                 0.0|
|   28|        3|2013-01-02| 4958|      609|   1|    0|           0|            1|          3931.5|              2479.0|
|   28|        4|2013-01-03| 5287|      641|   1|    0|           0|            1|          3559.2|              3415.0|
|   28|        5|2013-01-04| 5481|      623|   1|    0|           0|            1|          3559.2|              3931.5|
|   28|        6|2013-01-05| 2070|      287|   1|    0|           0|            0|          4592.8|              4449.0|
|   28|        7|2013-01-06|    

25/12/20 17:47:34 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 1057315 ms exceeds timeout 120000 ms
25/12/20 17:47:34 WARN SparkContext: Killing executors is not supported by current scheduler.
25/12/20 17:47:36 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:124)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$